### Chunking + embedding + similarity search + full-text search

In [117]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from src.utils.elastic_client import ElasticSearchClient
from src.utils.data_processor import  chunking_by_sentences
from src.utils.data_processor import embed, generate_stream
from src.utils.neo4j_client import Neo4jClient
import os
from dotenv import load_dotenv

load_dotenv()


True

In [3]:
ELASTIC_HOST = os.environ.get("ELASTIC_HOST")
API_KEY = os.environ.get('GENAI_API')
URI = os.environ.get('URI')
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD")

AUTH = (user, password)

In [55]:
es = ElasticSearchClient(ELASTIC_HOST)
es.get_info()

ObjectApiResponse({'name': 'e788d505d845', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'MFtuXAmyTy2S3RIzHPzhSA', 'version': {'number': '8.13.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'da95df118650b55a500dcc181889ac35c6d8da7c', 'build_date': '2024-05-06T22:04:45.107454559Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [56]:
doc = es.get_by_id(index_name='cafef-xahoi', doc_id='188260518095058352')
doc

{'title': 'Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị Loan',
 'detail_sapo': 'Nhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm.',
 'author': 'Duy Anh',
 'published_at': '2026-05-18-12-50',
 'content': 'TIN MỚI\nBộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ.\nÝ thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại.\nSau khi tiếp nhận thông tin, Đại úy Vũ Viết Liêm, cán bộ Công an xã đã báo cáo Ban chỉ huy Công an xã. Với tinh thần trách nhiệm cao và sự linh hoạt trong nghiệp vụ, lực lượng Công an xã đã nhanh chóng phối hợp với ngân hàng tiến hành xác minh nguồn dòng tiền.\nQua tra cứu, lực lượng chức năng đã xác định và l

In [114]:
content = '. '.join([doc['title'], doc['detail_sapo'], doc['content'].replace('\n', ' ')])
content

'Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị Loan. Nhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm.. TIN MỚI Bộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ. Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại. Sau khi tiếp nhận thông tin, Đại úy Vũ Viết Liêm, cán bộ Công an xã đã báo cáo Ban chỉ huy Công an xã. Với tinh thần trách nhiệm cao và sự linh hoạt trong nghiệp vụ, lực lượng Công an xã đã nhanh chóng phối hợp với ngân hàng tiến hành xác minh nguồn dòng tiền. Qua tra cứu, lực lượng chức năng đã xác định và liên hệ được với người chuyển nhầm tiền là chị Đỗ Thị Út Bốn (sinh năm 1968, trú tại tỉnh Lâm Đồng). Nhận 

## Chunking

In [58]:
article = content
result_chunks = chunking_by_sentences(article, chunk_size=3, overlap=1)

print(f"Total chunks: {len(result_chunks)}\n")
for idx, chunk in enumerate(result_chunks, 1):
    print(f"--- CHUNK {idx} ---")
    print(chunk)
    print()

Total chunks: 4

--- CHUNK 1 ---
Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị LoanNhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm. TIN MỚI Bộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ. Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại.

--- CHUNK 2 ---
Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại. Sau khi tiếp nhận thông tin, Đại úy Vũ Viết Liêm, cán bộ Công an xã đã báo cáo Ban chỉ huy Công an xã. Với tinh thần trách nhiệm cao và sự linh hoạt trong nghiệp vụ, lực lượng Công an xã đã nhanh chóng phối hợp với ngân hàng 

## Embedding

In [63]:
embeddings = embed(result_chunks, 'gemini-embedding-001')

In [66]:
len(embeddings[0])

3072

In [4]:
neo = Neo4jClient(URI, AUTH)
neo.open()

In [89]:
cypher = """
CREATE VECTOR INDEX cafef IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 3072,
    `vector.similarity_function`: 'cosine'
  }
}
"""

In [90]:
result = neo.query(cypher)
result

[]

In [92]:
cypher_query = '''
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
'''

In [93]:
query_params = {
    "chunks": result_chunks,       
    "embeddings": embeddings      
}

neo.query(cypher_query, parameters=query_params)


[]

In [97]:
query = "MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text"

In [101]:
record = neo.query(query)

In [108]:
record[0][0]["c.text"]

'Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị LoanNhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm. TIN MỚI Bộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ. Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại.'

In [138]:
# question = "Số tiền mà công an xác minh trong vụ giao dịch là bao nhiêu?"
# Số tiền mà công an xác minh trong vụ giao dịch là 2,9 tỷ đồng.
# question = "Tiền được chuyển vào tài khoản của ai?"
# Tiền được chuyển vào tài khoản của chị Lường Thị Loan.
# Chị Đỗ Thị Út Bốn đã vượt chặng đường hơn 1.800 km từ tỉnh Lâm Đồng đến tỉnh Lai Châu để phối hợp làm thủ tục.
question = "Thông tin cá nhân của chị Loan là gì?"
question_embed = embed([question])[0]

In [139]:
query = '''
CALL db.index.vector.queryNodes('cafef', 2, $question_embedding) 
YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''
config = {
    "question_embedding": question_embed
}

similar_records, _, _ = neo.query(query, config)
similar_records

[<Record text='Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại. Sau khi tiếp nhận thông tin, Đại úy Vũ Viết Liêm, cán bộ Công an xã đã báo cáo Ban chỉ huy Công an xã. Với tinh thần trách nhiệm cao và sự linh hoạt trong nghiệp vụ, lực lượng Công an xã đã nhanh chóng phối hợp với ngân hàng tiến hành xác minh nguồn dòng tiền.' score=0.8274116516113281 index=1>,
 <Record text='Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị LoanNhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm. TIN MỚI Bộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ. Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công 

In [140]:
for record in similar_records:
 print(record["text"])
 print(record["score"], record["index"])
 print("======")

Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại. Sau khi tiếp nhận thông tin, Đại úy Vũ Viết Liêm, cán bộ Công an xã đã báo cáo Ban chỉ huy Công an xã. Với tinh thần trách nhiệm cao và sự linh hoạt trong nghiệp vụ, lực lượng Công an xã đã nhanh chóng phối hợp với ngân hàng tiến hành xác minh nguồn dòng tiền.
0.8274116516113281 1
Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị LoanNhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm. TIN MỚI Bộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ. Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người tr

In [141]:
system_message = """
Bạn là một chabot hỏi đáp. 
Dựa vào các thông tin bạn có và được cung cấp, hãy trả lời câu hỏi một cách chính xác.
Nếu không chắc chắn thì nói không chắc.
"""
    
user_message = f"""
Dựa vào thông tin được cung cấp dưới đây, trả lời câu hỏi: {[doc["text"] for doc in similar_records]}
---
Câu hỏi cho đoạn thông tin trên: {question}
"""

In [142]:
print("Question:", question)
generate_stream(user_message, system_message, model='gemini-2.5-flash-lite')

Question: Thông tin cá nhân của chị Loan là gì?
Dựa vào thông tin được cung cấp, thông tin cá nhân của chị Loan là:

*   **Tên:** Lường Thị Loan
*   **Năm sinh:** 1989
*   **Địa chỉ:** Trú tại xã Pắc Ta, tỉnh Lai Châu

## Add full-text search -> hybrid

In [143]:
neo.query("CREATE FULLTEXT INDEX PdfChunkFulltext FOR (c:Chunk) ON EACH [c.text]")

([], <neo4j._work.summary.ResultSummary at 0x723adf1eee70>, [])

In [148]:
hybrid_query = '''
CALL {
 // vector index
 CALL db.index.vector.queryNodes('cafef', $k, $question_embedding)
YIELD node, score
 WITH collect({node:node, score:score}) AS nodes, max(score) AS max
 UNWIND nodes AS n
 // Normalize scores
 RETURN n.node AS node, (n.score / max) AS score
 UNION
 // keyword index
 CALL db.index.fulltext.queryNodes('PdfChunkFulltext', $question, {limit: $k})
 YIELD node, score
 WITH collect({node:node, score:score}) AS nodes, max(score) AS max
 UNWIND nodes AS n
 // Normalize scores
 RETURN n.node AS node, (n.score / max) AS score
}
// deduplicate nodes
WITH node, max(score) AS score ORDER BY score DESC LIMIT $k
RETURN node, score
'''


In [149]:
config = {
    "question_embedding": question_embed,
    "question": question,
    "k": 4
}

similar_hybrid_records, _, _ = neo.query(hybrid_query, config)
for record in similar_hybrid_records:
 print(record["node"]["text"])
 print(record["score"], record["node"]["index"])
 print("======")

Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại. Sau khi tiếp nhận thông tin, Đại úy Vũ Viết Liêm, cán bộ Công an xã đã báo cáo Ban chỉ huy Công an xã. Với tinh thần trách nhiệm cao và sự linh hoạt trong nghiệp vụ, lực lượng Công an xã đã nhanh chóng phối hợp với ngân hàng tiến hành xác minh nguồn dòng tiền.
1.0 1
Công an vào cuộc xác minh giao dịch 2,9 tỷ đồng gửi vào tài khoản của chị Lường Thị LoanNhờ sự vào cuộc xác minh nhanh chóng của lực lượng công an, số tài sản trên đã được bàn giao nguyên vẹn cho người chuyển nhầm. TIN MỚI Bộ Công an thông tin, vào ngày 8/5 vừa qua, chị Lường Thị Loan (sinh năm 1989, trú tại xã Pắc Ta, tỉnh Lai Châu) bất ngờ phát hiện tài khoản ngân hàng của mình nhận được số tiền 2,9 tỷ đồng từ một tài khoản xa lạ. Ý thức được đây là số tiền lớn không rõ nguồn gốc, chị Loan đã ngay lập tức đến trụ sở Công an xã Pắc Ta để trình báo và nhờ tìm người trả lại.
1.0 0
Nh

In [150]:
question

'Thông tin cá nhân của chị Loan là gì?'

In [6]:
neo.clear_database(db='neo4j')